# 02 — RAG com Validacao Semantica

Este notebook demonstra um pipeline de **Retrieval-Augmented Generation (RAG)** usando o `rag-corpus.jsonl` do Geolytics Dictionary. Para executar sem nenhuma chave de API usamos `rank_bm25` como mecanismo de recuperacao esparsa (BM25). Em producao substituiria pelo indice vetorial de sua escolha (FAISS, Chroma, Pinecone etc.).

Tambem demonstramos como o **validador semantico** do Geolytics captura alucinacoes tipicas, como confundir categorias de reservas SPE-PRMS.

## Pre-requisitos

```bash
pip install rank-bm25 requests
# Opcional para index vetorial:
# pip install faiss-cpu sentence-transformers
```

## Objetivo

1. Carregar e inspecionar o `rag-corpus.jsonl`.
2. Construir um indice BM25 como alternativa zero-config.
3. Demonstrar recuperacao correta: *'O que e Plano de Avaliacao de Descobertas?'*
4. Mostrar como um RAG vetor-only erra 'Reserva 4P' e como o validador corrige.
5. Visualizar pontuacoes dos candidatos de recuperacao.
6. Demonstrar a integracao com `scripts/validate-cli.js` via subprocess.

## 1. Importacoes

In [ ]:
import json
import math
import os
import re
import subprocess
import requests
from rank_bm25 import BM25Okapi
import matplotlib.pyplot as plt

print('rank-bm25 importado com sucesso')
# Saida esperada: rank-bm25 importado com sucesso

## 2. Carregamento do corpus

In [ ]:
CORPUS_URL = (
    'https://thiagoflc.github.io/geolytics-dictionary/ai/rag-corpus.jsonl'
)
LOCAL_CORPUS = os.path.join(
    os.path.dirname(os.getcwd()), 'ai', 'rag-corpus.jsonl'
)

corpus_docs = []

def _load_jsonl(text: str) -> list[dict]:
    docs = []
    for line in text.strip().splitlines():
        line = line.strip()
        if line:
            docs.append(json.loads(line))
    return docs

try:
    resp = requests.get(CORPUS_URL, timeout=15)
    resp.raise_for_status()
    corpus_docs = _load_jsonl(resp.text)
    print('Corpus carregado via HTTP')
except Exception as exc:
    print(f'HTTP falhou ({exc}), usando arquivo local...')
    with open(LOCAL_CORPUS, encoding='utf-8') as fh:
        corpus_docs = _load_jsonl(fh.read())
    print('Corpus carregado do disco')

print(f'Total de chunks: {len(corpus_docs)}')
# Inspeciona um chunk de exemplo
print('\nExemplo de chunk:')
print(json.dumps(corpus_docs[13], ensure_ascii=False, indent=2)[:400])
# Saida esperada:
# Corpus carregado via HTTP
# Total de chunks: 1245
# Exemplo de chunk: {"id": "term_pad", "type": "term", "text": "PAD...

## 3. Construcao do indice BM25

In [ ]:
# Tokenizacao simples: lowercase + split em espacos/pontuacao
def tokenize(text: str) -> list[str]:
    return re.sub(r'[^a-zA-Zaaccedeiioouuaeiou\s]', ' ', text.lower()).split()

texts = [doc['text'] for doc in corpus_docs]
tokenized_corpus = [tokenize(t) for t in texts]

bm25 = BM25Okapi(tokenized_corpus)
print(f'Indice BM25 construido com {len(tokenized_corpus)} documentos')
# Saida esperada:
# Indice BM25 construido com 1245 documentos

In [ ]:
def retrieve(query: str, top_k: int = 5) -> list[dict]:
    """Retorna os top_k chunks mais relevantes para a query."""
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    top_indices = sorted(range(len(scores)), key=lambda i: -scores[i])[:top_k]
    results = []
    for idx in top_indices:
        doc = corpus_docs[idx]
        results.append({
            'id':    doc.get('id', ''),
            'type':  doc.get('type', ''),
            'score': float(scores[idx]),
            'text':  doc.get('text', '')[:200],
        })
    return results

print('Funcao retrieve() definida')

## 4. Recuperacao correta: Plano de Avaliacao de Descobertas

In [ ]:
QUERY_PAD = 'O que e Plano de Avaliacao de Descobertas'

results_pad = retrieve(QUERY_PAD, top_k=5)

print(f'Query: "{QUERY_PAD}"')
print('=' * 60)
for i, r in enumerate(results_pad, 1):
    print(f'#{i}  score={r["score"]:.4f}  id={r["id"]}  tipo={r["type"]}')
    print(f'    {r["text"][:120]}...')
    print()
# Saida esperada:
# #1  score=...  id=term_pad  tipo=term
#     PAD — Plano de Avaliacao de Descobertas: Instrumento pelo qual o
# #2  score=...  id=term_declaracao-comercialidade  tipo=term
# ...

O resultado #1 e `term_pad`, que e exatamente o chunk correto. O BM25 acerta porque os tokens da query ('plano', 'avaliacao', 'descobertas') aparecem verbatim no texto do chunk.

## 5. Caso confuso: 'Reserva 4P'

Um RAG baseado apenas em similaridade vetorial pode confundir 'Reserva 4P' com 'Reserva Ambiental' (4P = categoria ambiental no contexto fundiario brasileiro). O validador semantico captura essa confusao verificando o contexto do dominio O&G vs. ambiental.

In [ ]:
QUERY_4P = 'O que e Reserva 4P'

results_4p = retrieve(QUERY_4P, top_k=5)

print(f'Query: "{QUERY_4P}"')
print('=' * 60)
for i, r in enumerate(results_4p, 1):
    print(f'#{i}  score={r["score"]:.4f}  id={r["id"]}  tipo={r["type"]}')
    print(f'    {r["text"][:160]}...')
    print()
# Saida esperada:
# Os resultados podem incluir chunks sobre SPE-PRMS (reservas 1P/2P/3P)
# mas dificilmente '4P' pois nao e uma categoria SPE-PRMS valida —
# esse e exatamente o comportamento incorreto que o validador detecta.

In [ ]:
# Simulacao de uma resposta errada que um LLM poderia gerar
FAKE_ANSWER_4P = (
    'Reserva 4P e a quarta categoria de reservas da classificacao SPE-PRMS, '
    'correspondendo a recursos contingentes de menor probabilidade de '
    'desenvolvimento comercial.'
)

print('Resposta hipotetica do LLM (sem validacao):')
print(FAKE_ANSWER_4P)

print()
print('PROBLEMA: A classificacao SPE-PRMS nao possui categoria 4P.')
print('As categorias sao: 1P (provadas), 2P (provadas+provaveis), 3P (provadas+provaveis+possiveis).')
print('"4P" nao e um termo tecnico valido na industria de O&G.')

### 5.1 Validacao via subprocess (validate-cli.js)

In [ ]:
# O validate-cli.js e o validador semantico do Geolytics (Node.js).
# Podemos chama-lo via subprocess a partir do Python.

VALIDATOR_PATH = os.path.join(
    os.path.dirname(os.getcwd()), 'scripts', 'validate-cli.js'
)

claim_to_validate = {
    'term': '4P',
    'answer': FAKE_ANSWER_4P,
    'context': 'SPE-PRMS reservas'
}

if os.path.exists(VALIDATOR_PATH):
    try:
        proc = subprocess.run(
            ['node', VALIDATOR_PATH, json.dumps(claim_to_validate)],
            capture_output=True,
            text=True,
            timeout=10,
        )
        print('Saida do validador:')
        print(proc.stdout or proc.stderr)
    except Exception as exc:
        print(f'Nao foi possivel executar o validador: {exc}')
else:
    print(f'validate-cli.js nao encontrado em {VALIDATOR_PATH}')
    print('Para validar, execute a partir do diretorio raiz do repositorio.')
# Saida esperada (se node disponivel):
# { valid: false, violations: [{ rule: 'spe-prms-categories', ... }] }

## 6. Visualizacao das pontuacoes de recuperacao

In [ ]:
queries = {
    'PAD (correto)': QUERY_PAD,
    'Reserva 4P (ambiguo)': QUERY_4P,
    'Regime contratual': 'O que e regime contratual de concessao',
}

fig, axes = plt.subplots(1, len(queries), figsize=(14, 4))

for ax, (title, query) in zip(axes, queries.items()):
    results = retrieve(query, top_k=8)
    ids = [r['id'].replace('term_', '')[:18] for r in results]
    scores = [r['score'] for r in results]
    ax.barh(ids[::-1], scores[::-1], color='#378ADD')
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Score BM25')
    ax.tick_params(labelsize=7)

plt.suptitle('Pontuacoes BM25 por Query', fontsize=11)
plt.tight_layout()
plt.show()
# Saida esperada: tres graficos de barras horizontais mostrando
# os chunks recuperados e seus scores BM25 para cada query.

## 7. Substituicao por indice vetorial (opcional)

Para usar FAISS com embeddings locais (sem API), instale:

```bash
pip install faiss-cpu sentence-transformers
```

E substitua `retrieve()` pela versao abaixo:

In [ ]:
# ---------------------------------------------------------------------------
# CELULA OPCIONAL — nao e executada automaticamente
# ---------------------------------------------------------------------------
# from sentence_transformers import SentenceTransformer
# import faiss
# import numpy as np
#
# model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
# embeddings = model.encode(texts, show_progress_bar=True)
# embeddings = np.array(embeddings).astype('float32')
# faiss.normalize_L2(embeddings)
#
# index = faiss.IndexFlatIP(embeddings.shape[1])
# index.add(embeddings)
#
# def retrieve_vec(query: str, top_k: int = 5):
#     q_emb = model.encode([query]).astype('float32')
#     faiss.normalize_L2(q_emb)
#     distances, indices = index.search(q_emb, top_k)
#     return [{'id': corpus_docs[i]['id'], 'score': float(d)}
#              for i, d in zip(indices[0], distances[0])]
print('Celula opcional — descomente para usar FAISS com embeddings locais')

## Proximo passo

Veja o notebook **03_langgraph_multiagent.ipynb** para integrar o RAG ao agente LangGraph multi-no com roteamento semantico.